# Heart Disease Prediction using Machine Learning
> **Diagnostic Classification using Logistic Regression (Cleveland Heart Disease Dataset)**

---

### Project Overview
Cardiovascular diseases (CVDs) are the leading cause of mortality globally. Identifying high-risk individuals through clinical diagnostics enables early intervention, lifestyle modification, and targeted treatment to prevent severe cardiac events.

This notebook develops an interpretable, production-ready machine learning classification model to predict the presence of heart disease based on 13 patient diagnostic measurements from the **Cleveland Heart Disease Dataset**.

### Key Workflow Highlights:
- **Clinical Feature Analysis**: Evaluates 13 key cardiovascular risk factors including resting blood pressure (`trestbps`), serum cholesterol (`chol`), maximum heart rate achieved (`thalach`), ST depression induced by exercise (`oldpeak`), chest pain type (`cp`), and fluoroscopy vessel counts (`ca`).
- **Stratified Validation Strategy**: Uses stratified splitting and **5-Fold Stratified Cross-Validation** to ensure consistent disease prevalence ratios across all evaluation folds.
- **Comprehensive Benchmarking**: Evaluates model performance across Accuracy, Precision, Recall (Sensitivity), F1-Score, ROC-AUC, Confusion Matrix, and full Classification Report.

---

## 1. Imports and Environment Setup
Import essential libraries for numerical processing, data analysis, statistical modeling, cross-validation, and performance evaluation.

In [1]:
import numpy as np
import pandas as pd
import pickle

# Model & Selection
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold
)

# Performance Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Display configuration
pd.set_option('display.max_columns', None)


## 2. Data Ingestion and Exploratory Data Analysis (EDA)
Load the heart disease dataset and inspect its structure, dimensions, data types, missing values, and summary statistics.

In [2]:
# Load the dataset
heart_data = pd.read_csv('../Dataset/heart.csv')

print(f"Dataset Dimensions: {heart_data.shape[0]} rows x {heart_data.shape[1]} columns")
print(f"Missing Values: {heart_data.isnull().sum().sum()} total across all columns\n")
heart_data.head()


Dataset Dimensions: 303 rows x 14 columns
Missing Values: 0 total across all columns



,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [3]:
# Dataset schema and data types
heart_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


In [4]:
# Statistical summary of clinical features
heart_data.describe().T


,count,mean,std,min,25%,50%,75%,max
age,303.0,54.366337,9.082101,29.0,47.5,55.0,61.0,77.0
sex,303.0,0.683168,0.466011,0.0,0.0,1.0,1.0,1.0
cp,303.0,0.966997,1.032052,0.0,0.0,1.0,2.0,3.0
trestbps,303.0,131.623762,17.538143,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.264026,51.830751,126.0,211.0,240.0,274.5,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.528053,0.525860,0.0,0.0,1.0,1.0,2.0
thalach,303.0,149.646865,22.905161,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


## 3. Target Class Distribution & Feature Comparisons
Analyze the distribution of the diagnostic target variable (`target: 1` = Heart Disease Present / Defective Heart, `target: 0` = Healthy Heart) and compare mean feature values between groups.

In [5]:
# Target distribution
print("=== Class Distribution (Diagnostic Target) ===")
target_counts = heart_data['target'].value_counts()
print(f"Heart Disease Present (1) : {target_counts[1]} ({target_counts[1]/len(heart_data)*100:.1f}%)")
print(f"Healthy Heart         (0) : {target_counts[0]} ({target_counts[0]/len(heart_data)*100:.1f}%)")

# Mean feature comparison by disease status
print("\n=== Mean Clinical Values Grouped by Heart Disease Status ===")
display(heart_data.groupby('target').mean().T.rename(columns={0: 'Healthy Heart (0)', 1: 'Heart Disease (1)'}))


=== Class Distribution (Diagnostic Target) ===
Heart Disease Present (1) : 165 (54.5%)
Healthy Heart         (0) : 138 (45.5%)

=== Mean Clinical Values Grouped by Heart Disease Status ===


target,Healthy Heart (0),Heart Disease (1)
age,56.601449,52.496970
sex,0.826087,0.563636
cp,0.478261,1.375758
trestbps,134.398551,129.303030
chol,251.086957,242.230303
fbs,0.159420,0.139394
restecg,0.449275,0.593939
thalach,139.101449,158.466667
exang,0.550725,0.139394
oldpeak,1.585507,0.583030


## 4. Feature and Target Preparation
Separate the input feature matrix (`X`) from the diagnostic target variable (`Y`).

In [6]:
# Separate feature matrix (X) and target variable (Y)
X = heart_data.drop(columns='target')
Y = heart_data['target']

print(f"Feature Matrix Shape : {X.shape}")
print(f"Target Vector Shape  : {Y.shape}")
print(f"Clinical Features    : {list(X.columns)}")


Feature Matrix Shape : (303, 13)
Target Vector Shape  : (303,)
Clinical Features    : ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


## 5. Stratified Train/Test Split
Partition the data into an 80% training set and a 20% test set using stratified sampling to preserve the diagnostic class balance across both splits.

In [7]:
# Stratified 80/20 train/test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    stratify=Y,
    random_state=2
)

print(f"Total Patient Records : {X.shape[0]}")
print(f"Training Set          : {X_train.shape[0]} samples (Class balance: {Y_train.value_counts().to_dict()})")
print(f"Test Set              : {X_test.shape[0]} samples (Class balance: {Y_test.value_counts().to_dict()})")


Total Patient Records : 303
Training Set          : 242 samples (Class balance: {1: 132, 0: 110})
Test Set              : 61 samples (Class balance: {1: 33, 0: 28})


## 6. Model Architecture & Training

### Logistic Regression
Logistic Regression is well-suited for binary clinical diagnosis because it provides interpretable log-odds coefficients and well-calibrated posterior probabilities. We set `max_iter=1000` to guarantee optimization convergence.

In [8]:
# Instantiate Logistic Regression model
model = LogisticRegression(max_iter=1000)

# Fit model on training data
model.fit(X_train, Y_train)
print("Logistic Regression model successfully fitted on training data.")


Logistic Regression model successfully fitted on training data.


## 7. Comprehensive Model Evaluation

We evaluate model performance through:
1. **Training Accuracy**: Baseline verification of model fit on training data.
2. **Held-Out Test Set**: Evaluation on unseen test patients across standard clinical metrics.
3. **Confusion Matrix & Classification Report**: Detailed breakdown of true positives, true negatives, sensitivity, and specificity.

In [9]:
# 1. Training Set Accuracy
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(Y_train, X_train_prediction)
print(f"Accuracy on Training data : {training_data_accuracy * 100:.2f}%")


Accuracy on Training data : 85.54%


In [10]:
# 2. Comprehensive Test Set Evaluation
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(Y_test, X_test_prediction)

precision = precision_score(Y_test, X_test_prediction)
recall = recall_score(Y_test, X_test_prediction)
f1 = f1_score(Y_test, X_test_prediction)
y_probs = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(Y_test, y_probs)

print("=" * 55)
print("           HELD-OUT TEST SET EVALUATION")
print("=" * 55)
print(f"  Accuracy            : {test_data_accuracy * 100:.2f}%")
print(f"  Precision           : {precision:.4f}")
print(f"  Recall (Sensitivity): {recall:.4f}")
print(f"  F1-Score            : {f1:.4f}")
print(f"  ROC-AUC Score       : {roc_auc:.4f}")
print("=" * 55)

# Confusion Matrix
cm = confusion_matrix(Y_test, X_test_prediction)
print("\n--- Confusion Matrix ---")
print(cm)
print(f"  True Negatives (Healthy correctly classified)     : {cm[0,0]}")
print(f"  False Positives (Healthy misclassified as disease): {cm[0,1]}")
print(f"  False Negatives (Disease missed by model)         : {cm[1,0]}")
print(f"  True Positives (Disease correctly identified)     : {cm[1,1]}")

# Classification Report
print("\n--- Detailed Classification Report ---")
print(classification_report(Y_test, X_test_prediction, target_names=['Healthy Heart (0)', 'Defective Heart (1)']))


           HELD-OUT TEST SET EVALUATION
  Accuracy            : 80.33%
  Precision           : 0.8182
  Recall (Sensitivity): 0.8182
  F1-Score            : 0.8182
  ROC-AUC Score       : 0.9037

--- Confusion Matrix ---
[[22  6]
 [ 6 27]]
  True Negatives (Healthy correctly classified)     : 22
  False Positives (Healthy misclassified as disease): 6
  False Negatives (Disease missed by model)         : 6
  True Positives (Disease correctly identified)     : 27

--- Detailed Classification Report ---
                     precision    recall  f1-score   support

  Healthy Heart (0)       0.79      0.79      0.79        28
Defective Heart (1)       0.82      0.82      0.82        33

           accuracy                           0.80        61
          macro avg       0.80      0.80      0.80        61
       weighted avg       0.80      0.80      0.80        61



## 8. 5-Fold Stratified Cross-Validation
Validate the model's generalization performance across the full dataset using **5-Fold `StratifiedKFold`** to ensure robustness against split variance.

In [11]:
# 5-Fold Stratified Cross-Validation on the full dataset
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

cv_acc = cross_val_score(model, X, Y, cv=cv, scoring='accuracy')
cv_precision = cross_val_score(model, X, Y, cv=cv, scoring='precision')
cv_recall = cross_val_score(model, X, Y, cv=cv, scoring='recall')
cv_f1 = cross_val_score(model, X, Y, cv=cv, scoring='f1')
cv_roc = cross_val_score(model, X, Y, cv=cv, scoring='roc_auc')

print("=" * 55)
print("   5-FOLD STRATIFIED CROSS-VALIDATION RESULTS")
print("=" * 55)
print(f"  Accuracy : {cv_acc.mean()*100:.2f}% (+/- {cv_acc.std()*100:.2f}%)")
print(f"  Precision: {cv_precision.mean():.4f} (+/- {cv_precision.std():.4f})")
print(f"  Recall   : {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})")
print(f"  F1-Score : {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")
print(f"  ROC-AUC  : {cv_roc.mean():.4f} (+/- {cv_roc.std():.4f})")
print("=" * 55)


   5-FOLD STRATIFIED CROSS-VALIDATION RESULTS
  Accuracy : 82.22% (+/- 6.01%)
  Precision: 0.8090 (+/- 0.0542)
  Recall   : 0.8848 (+/- 0.0727)
  F1-Score : 0.8436 (+/- 0.0535)
  ROC-AUC  : 0.8895 (+/- 0.0378)


## 9. Building a Predictive System
Demonstrate real-time inference using the trained Logistic Regression model on a representative patient diagnostic vector.

In [12]:
# Sample patient diagnostic measurements (13 clinical features)
# [age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal]
input_data = (62, 0, 0, 140, 268, 0, 0, 160, 0, 3.6, 0, 2, 2)

# Convert to DataFrame with feature column names
input_df = pd.DataFrame([input_data], columns=X.columns)

# Generate prediction and predicted probabilities
prediction = model.predict(input_df)
predicted_proba = model.predict_proba(input_df)[0]

print(f"Predicted Class        : {prediction[0]}")
print(f"Disease Probability    : {predicted_proba[1] * 100:.2f}%")
print(f"Healthy Probability    : {predicted_proba[0] * 100:.2f}%")

if prediction[0] == 0:
    print("Diagnostic Outcome     : The person does not have heart disease (Healthy Heart)")
else:
    print("Diagnostic Outcome     : The person has heart disease (Defective Heart)")


Predicted Class        : 0
Disease Probability    : 11.40%
Healthy Probability    : 88.60%
Diagnostic Outcome     : The person does not have heart disease (Healthy Heart)


## 10. Saving and Verifying the Trained Model
Serialize the trained Logistic Regression model to disk and verify that it loads and predicts correctly for deployment in the Streamlit application.

In [13]:
# Save trained model to disk
model_filename = '../saved_models/heart_disease_model.sav'
pickle.dump(model, open(model_filename, 'wb'))
print(f"Model successfully saved to '{model_filename}'")

# Verification: Load the saved model and test prediction consistency
loaded_model = pickle.load(open(model_filename, 'rb'))
verification_pred = loaded_model.predict(input_df)
print(f"Verification Test with Loaded Model: Prediction = {verification_pred[0]} (Match: {verification_pred[0] == prediction[0]})")


Model successfully saved to '../saved_models/heart_disease_model.sav'
Verification Test with Loaded Model: Prediction = 0 (Match: True)
